In [109]:
import numpy as np
import pandas as pd


In [110]:
df = pd.read_csv('IMDB_Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [111]:
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

### Text Cleaning
1. Sample 10000 rows
2. Remove HTML Tags
3. Remove special characcters
4. Converting every thing in lower case
5. removeing stop words
6. stemming

In [112]:
df = df.sample(10000)
df.shape

(10000, 2)

In [113]:
df['sentiment'] = df['sentiment'].replace({'positive': 1, 'negative': 0})
df.head()

,review,sentiment
33255,Centres on Czech WW2 pilots  the older Franti...,1
22648,What's the matter with you people? John Dahl? ...,1
41949,I found the film quite good for what I was exp...,1
30609,A harrowing masterpiece on the sheer madness a...,1
11566,This film limps from self indulgent moment to ...,0


In [114]:
import re
clean = re.compile('<.*?>')
re.sub(clean, '', df.iloc[2].review)

"I found the film quite good for what I was expecting. Although I weary, because I have a fear of injection needles, I sort of came to expect when they were coming. So if you're not into needles, blood, the human body, and some good medical fun, put this movie back and rent another. As the other user commented, I was also please at the German attempt at a slasher film. I'm an American who just moved to Germany to stay with a family and saw this lying on the shelf. I love psychological thrillers, and I'd say this is somewhere along those lines. A character falls into places and feels misconstrued. While trying to dig her way out and find some truth to a situation, things get a little sticky and other aren't so sure she's on the right track. So throughout the film you're kept on edge about who's anatomy you might catch a glimpse of and who's rounding the next corner."

In [115]:
# fuction to clean html tags
def clean_html(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '',text)

In [116]:
df['review'] = df['review'].apply(clean_html)

In [117]:
# converting everthing in lower
def convert_lower(text):
    return text.lower()
df['review'] = df['review'].apply(convert_lower)

In [118]:
# to remove specal charactors
def remove_special(text):
    x = ''

    for i in text:
        if i.isalnum():
            x = x+i
        else:
            x = x + ' '
    return x

df['review'] = df['review'].apply(remove_special)

In [119]:
import nltk
from nltk.corpus import stopwords

# 1. Download only if you haven't already
nltk.download('stopwords', quiet=True) 

# 2. Create a SET of stopwords outside the function for instant lookups
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    # 3. Use a list comprehension to build and return the list in one line,
    # ensuring we check the lowercase version of the word.
    return [word for word in text.split() if word.lower() not in stop_words]

# --- Example Usage ---
# sample_text = "The quick brown fox jumps over the lazy dog"
# print(remove_stopwords(sample_text))
# Output: ['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog']

In [120]:
df['review'] =df['review'].apply(remove_stopwords)
df.head()

,review,sentiment
33255,"[centres, czech, ww2, pilots, older, frantisek...",1
22648,"[matter, people, john, dahl, rounders, unforge...",1
41949,"[found, film, quite, good, expecting, although...",1
30609,"[harrowing, masterpiece, sheer, madness, despa...",1
11566,"[film, limps, self, indulgent, moment, self, i...",0


In [121]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stem_words(text):
    # 1. If the previous step turned the text into a LIST, just stem and join
    if isinstance(text, list):
        return " ".join([ps.stem(word) for word in text])
    
    # 2. If it is still a standard STRING, split, stem, and join
    elif isinstance(text, str):
        return " ".join([ps.stem(word) for word in text.split()])
    
    # 3. If the row is empty or NaN, return an empty string to prevent crashing
    else:
        return ""

In [122]:
df['review'] =df['review'].apply(stem_words)
df.head()

,review,sentiment
33255,centr czech ww2 pilot older frantisek boyish i...,1
22648,matter peopl john dahl rounder unforgett quirk...,1
41949,found film quit good expect although weari fea...,1
30609,harrow masterpiec sheer mad despair war fire p...,1
11566,film limp self indulg moment self indulg momen...,0


In [123]:
X = df.iloc[:,0:1].values
X.shape

(10000, 1)

In [144]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=10000)

In [145]:
X = cv.fit_transform(df['review']).toarray()
X.shape

(10000, 10000)

In [147]:
y = df.iloc[:,-1].values
y.shape

(10000,)

In [148]:
# X,y
# Training set
# test set(alredy know the result)

In [149]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,)

In [150]:
X_train.shape

(8000, 10000)

In [151]:
from sklearn.naive_bayes import GaussianNB,MultinomialNB,BernoulliNB
y_train = y_train.astype('int')
y_test = y_test.astype('int')
clf1 = GaussianNB()
clf2 = MultinomialNB()
clf3 = BernoulliNB()

clf1.fit(X_train,y_train)
clf2.fit(X_train,y_train)
clf3.fit(X_train,y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"binarize binarize: float or None, default=0.0Threshold for binarizing (mapping to booleans) of sample features.If None, input is presumed to already consist of binary vectors.",0.0
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[3935.,4065.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Log probability of each class (smoothed).","ndarray[float64](2,)","[-0.71,-0.68]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 10000)","[[13.,44., 2.,..., 5., 0., 2.], [14.,48., 6.,..., 4., 1., 2.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of features given a class, P(x_i|y).","ndarray[float64](2, 10000)","[[-5.64,-4.47,-7.18,...,-6.49,-8.28,-7.18], [-5.6 ,-4.42,-6.36,...,-6.7 ,-7.62,-7.21]]"


In [152]:
y_pred1 = clf1.predict(X_test)
y_pred2 = clf2.predict(X_test)
y_pred3 = clf3.predict(X_test)

In [153]:
from sklearn.metrics import accuracy_score
print("GaussianNB: ",accuracy_score(y_test,y_pred1))
print("MultinomialNB: ",accuracy_score(y_test,y_pred2))
print("BernoulliNB: ",accuracy_score(y_test,y_pred3))

GaussianNB:  0.636
MultinomialNB:  0.839
BernoulliNB:  0.843
